# BERT на DUSHA: speaker_text → ruBERT-base fine-tune

Pipeline:
1. Читаем `speaker_text` из `aggregated_ds_0.9.tsv` (Dawid-Skene 0.9)
2. Fine-tune `DeepPavlov/rubert-base-cased` (5 классов DUSHA)

Val (15% от train-выборки) подготавливается один раз перед обучением.  
Eval каждые **2500 шагов**. Лучшая модель сохраняется по **val_wacc**.  
Лучшая модель сохраняется в `/kaggle/working/`.

## 1. Install & clone

In [ ]:
import subprocess, sys, os

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'datasets', 'pyyaml', 'tqdm', 'accelerate',
], check=True)

REPO_URL = 'https://github.com/aibryanov/speech_emo_finetune.git'
REPO_DIR = 'speech_emo_finetune'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
os.chdir(REPO_DIR)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('Done. CWD:', os.getcwd())

## 2. Imports & config

In [ ]:
import warnings, pathlib, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

# ── пути ──────────────────────────────────────────────────────────────────────
AGG_ROOT   = pathlib.Path('/kaggle/input/datasets/aleksandribryanov/agg-dusha')
TRAIN_TSV  = AGG_ROOT / 'aggregated_ds_0.9.tsv'
TEST_TSV   = AGG_ROOT / 'aggregated_ds_0.9_test.tsv'
OUT_DIR    = pathlib.Path('/kaggle/working')

# ── гиперпараметры ────────────────────────────────────────────────────────────
TRAIN_FRACTION = 0.05   # доля train TSV (~7k записей)
VAL_FRACTION   = 0.15   # доля под валидацию
BERT_BATCH     = 32
LR             = 2e-5
WEIGHT_DECAY   = 1e-2
WARMUP_STEPS   = 200
MAX_STEPS      = 20_000
EVAL_EVERY     = 2500
ES_PATIENCE    = 5
MAX_SEQ_LEN    = 128
SEED           = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

DUSHA_LABEL2ID = {'neutral': 0, 'angry': 1, 'positive': 2, 'sad': 3, 'other': 4}
DUSHA_LABELS   = ['neutral', 'angry', 'positive', 'sad', 'other']

## 3. Загрузка данных из TSV (speaker_text)

In [ ]:
from sklearn.model_selection import train_test_split

def load_tsv(tsv_path, fraction=None):
    df = pd.read_csv(tsv_path, sep='\t')
    df = df[df['aggregated_emo'].isin(DUSHA_LABEL2ID)]
    df = df[df['speaker_text'].notna() & (df['speaker_text'].str.strip() != '')]
    if fraction:
        df = df.sample(frac=fraction, random_state=SEED)
    texts  = df['speaker_text'].tolist()
    labels = [DUSHA_LABEL2ID[e] for e in df['aggregated_emo']]
    print(f'  {tsv_path.name}: {len(texts)} records')
    return texts, labels

train_texts_all, train_labels_all = load_tsv(TRAIN_TSV, fraction=TRAIN_FRACTION)
test_texts,      test_labels      = load_tsv(TEST_TSV)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts_all, train_labels_all,
    test_size=VAL_FRACTION, random_state=SEED,
    stratify=train_labels_all,
)

from collections import Counter
print(f'\nTrain: {len(train_texts)}  Val: {len(val_texts)}  Test: {len(test_texts)}')
for lid, name in enumerate(DUSHA_LABELS):
    t = sum(1 for l in train_labels if l == lid)
    v = sum(1 for l in val_labels   if l == lid)
    print(f'  {name:10s}  train={t:4d}  val={v:4d}')

print(f'\nПример: "{train_texts[0]}"  → {DUSHA_LABELS[train_labels[0]]}')

## 5. BERT модель

In [ ]:
BERT_MODEL = 'DeepPavlov/rubert-base-cased'

print(f'Loading BERT: {BERT_MODEL}')
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
bert = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL,
    num_labels=len(DUSHA_LABELS),
    problem_type='single_label_classification',
).to(device)

total = sum(p.numel() for p in bert.parameters())
print(f'Parameters: {total:,}')
print('Label mapping:', {i: l for i, l in enumerate(DUSHA_LABELS)})


class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_SEQ_LEN):
        self.encodings = tokenizer(
            texts, truncation=True, padding='max_length',
            max_length=max_len, return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {
            'input_ids':      self.encodings['input_ids'][i],
            'attention_mask': self.encodings['attention_mask'][i],
            'labels':         self.labels[i],
        }


print('Tokenizing...')
train_ds = TextDataset(train_texts, train_labels, tokenizer)
val_ds   = TextDataset(val_texts,   val_labels,   tokenizer)
train_ld = DataLoader(train_ds, batch_size=BERT_BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_ld   = DataLoader(val_ds,   batch_size=BERT_BATCH, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train batches: {len(train_ld)}  Val batches: {len(val_ld)}')

## 6. Обучение

In [ ]:
from sklearn.metrics import balanced_accuracy_score

optimizer = torch.optim.AdamW(bert.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler_warmup = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=MAX_STEPS)
scheduler_plateau = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2)  # mode=max для wacc

best_val_wacc = -1.0
es_counter    = 0
global_step   = 0
history       = []
train_iter    = iter(train_ld)


@torch.no_grad()
def evaluate():
    bert.eval()
    total_loss, preds_all, labels_all = 0.0, [], []
    for batch in val_ld:
        batch = {k: v.to(device) for k, v in batch.items()}
        out   = bert(**batch)
        total_loss  += out.loss.item() * batch['labels'].size(0)
        preds_all.append(out.logits.argmax(-1).cpu().numpy())
        labels_all.append(batch['labels'].cpu().numpy())
    val_loss = total_loss / len(val_ds)
    val_wacc = balanced_accuracy_score(
        np.concatenate(labels_all), np.concatenate(preds_all))
    bert.train()
    return val_loss, val_wacc


print(f'Starting training  max_steps={MAX_STEPS}  eval_every={EVAL_EVERY}  es_patience={ES_PATIENCE}')
print(f'LR={LR}  batch={BERT_BATCH}  warmup={WARMUP_STEPS}  save_by=val_wacc')
print('-' * 70)

bert.train()
running_loss = 0.0

while global_step < MAX_STEPS:
    try:
        batch = next(train_iter)
    except StopIteration:
        train_iter = iter(train_ld)
        batch = next(train_iter)

    batch = {k: v.to(device) for k, v in batch.items()}
    optimizer.zero_grad()
    out  = bert(**batch)
    loss = out.loss
    loss.backward()
    nn.utils.clip_grad_norm_(bert.parameters(), 1.0)
    optimizer.step()
    scheduler_warmup.step()
    running_loss += loss.item()
    global_step  += 1

    if global_step % EVAL_EVERY == 0:
        avg_train_loss = running_loss / EVAL_EVERY
        val_loss, val_wacc = evaluate()
        prev_lr = optimizer.param_groups[0]['lr']
        scheduler_plateau.step(val_wacc)          # scheduler смотрит на wacc
        cur_lr  = optimizer.param_groups[0]['lr']
        running_loss = 0.0

        is_best = val_wacc > best_val_wacc        # сохраняем по wacc
        if is_best:
            best_val_wacc = val_wacc
            es_counter    = 0
            bert.save_pretrained(str(OUT_DIR / 'best_bert_dusha'))
            tokenizer.save_pretrained(str(OUT_DIR / 'best_bert_dusha'))
        else:
            es_counter += 1

        lr_info = f'{cur_lr:.2e}' + (' ↓' if cur_lr < prev_lr else '')
        history.append((global_step, avg_train_loss, val_loss, val_wacc))
        print(
            f'Step {global_step:6d}  train_loss={avg_train_loss:.4f}  '
            f'val_loss={val_loss:.4f}  val_wacc={val_wacc:.4f}  '
            f'lr={lr_info}' + ('  *' if is_best else ''),
            flush=True,
        )

        if es_counter >= ES_PATIENCE:
            print(f'Early stopping at step {global_step}')
            break

print(f'\nBest val_wacc: {best_val_wacc:.4f}')
print(f'Model saved → {OUT_DIR / "best_bert_dusha"}')

## 7. Кривые обучения

In [ ]:
import matplotlib.pyplot as plt

steps      = [h[0] for h in history]
tr_losses  = [h[1] for h in history]
val_losses = [h[2] for h in history]
val_waccs  = [h[3] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(steps, tr_losses,  label='train loss', color='steelblue')
ax1.plot(steps, val_losses, label='val loss',   color='darkorange')
best_step = steps[int(np.argmin(val_losses))]
ax1.axvline(best_step, color='red', linestyle='--', alpha=0.6, label=f'best step {best_step}')
ax1.set_xlabel('Step'); ax1.set_ylabel('Loss')
ax1.set_title('Train / Val Loss'); ax1.legend()

ax2.plot(steps, val_waccs, color='green', label='val wacc')
ax2.axvline(best_step, color='red', linestyle='--', alpha=0.6)
ax2.set_xlabel('Step'); ax2.set_ylabel('Weighted Accuracy')
ax2.set_title('Val Weighted Accuracy'); ax2.legend()

plt.tight_layout()
plt.savefig(str(OUT_DIR / 'bert_training_curves.png'), dpi=150)
plt.show()

## 8. Финальная оценка на test TSV

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, balanced_accuracy_score

bert_best = AutoModelForSequenceClassification.from_pretrained(
    str(OUT_DIR / 'best_bert_dusha')).to(device)
bert_best.eval()

test_ds = TextDataset(test_texts, test_labels, tokenizer)
test_ld = DataLoader(test_ds, batch_size=BERT_BATCH, shuffle=False, num_workers=2)

preds_all, labels_all = [], []
with torch.no_grad():
    for batch in tqdm(test_ld, desc='Test eval'):
        batch = {k: v.to(device) for k, v in batch.items()}
        out   = bert_best(**batch)
        preds_all.append(out.logits.argmax(-1).cpu().numpy())
        labels_all.append(batch['labels'].cpu().numpy())

preds  = np.concatenate(preds_all)
labels = np.concatenate(labels_all)

print('\n=== Test Results ===')
print(f'Accuracy          : {accuracy_score(labels, preds):.4f}')
print(f'Weighted Accuracy  : {balanced_accuracy_score(labels, preds):.4f}')
print()
print(classification_report(labels, preds, target_names=DUSHA_LABELS, zero_division=0))